# ML Zoomcamp 2026 — Homework 5: Deploying Machine Learning Models

This homework loads the frozen `pipeline.bin` and reports the values the submission form asks for. The released model is used as-is, and each question below records the value to submit. Official questions are in [materials/homework.md](materials/homework.md).


In [1]:
# import os: read and set environment variables for this process.
import os

# setdefault: set the name only when it is missing.
# MPLCONFIGDIR: Matplotlib config directory. Use the E: cache instead of the C: user profile.
os.environ.setdefault("MPLCONFIGDIR", r"E:\IT_SPACES\AI\.cache\matplotlib")
# XDG_CACHE_HOME: shared cache root.
os.environ.setdefault("XDG_CACHE_HOME", r"E:\IT_SPACES\AI\.cache")
# TEMP, TMP: temporary files. Both point at the same E: tmp directory.
os.environ.setdefault("TEMP", r"E:\IT_SPACES\AI\.cache\tmp")
os.environ.setdefault("TMP", r"E:\IT_SPACES\AI\.cache\tmp")
# OMP/MKL/OPENBLAS/NUMEXPR: keep numpy and scikit-learn on one thread.
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")

print("cache paths set")


cache paths set


## Q1 — environment check

**Question:** Run `uv --version` and record the version in your notes. This is a setup check, not a graded answer: your local `uv` version is not a property of the model.

```bash
uv --version
```

**Answer:** `uv 0.13.5 (46776a95d 2026-10-21 x86_64-pc-windows-msvc)`


In [2]:
# import subprocess: run an external command and capture its output.
import subprocess

# subprocess.run: Question 1 runs uv --version. The result is not graded.
# capture_output=True, text=True: read the version line as text.
try:
    completed = subprocess.run(
        ["uv", "--version"],
        capture_output=True,
        text=True,
        timeout=8,
    )
    # If stdout is empty, show stderr.
    print((completed.stdout or completed.stderr).strip())
except subprocess.TimeoutExpired:
    print("uv --version timed out")


uv 0.13.5 (46776a95d 2026-10-21 x86_64-pc-windows-msvc)


## Q2 — locked dependency

**Question:** Open `pyproject.toml` and `uv.lock`. Which Scikit-Learn version is part of the 2026 reference environment?

- `1.6.1`
- `1.7.2`
- `1.8.0`
- `2.0.0`

The graded value is the lockfile version. The shared venv’s installed package is 1.9.0.

**Answer:** **1.7.2**


In [3]:
# import re: find the version string in a file.
import re
# Path: locations of pyproject.toml and uv.lock.
from pathlib import Path

# cwd: the folder Jupyter used when this notebook was opened, which is HW_05.
root = Path.cwd()
# read_text: read the existing pins. Do not regenerate the lockfile.
project_text = (root / "pyproject.toml").read_text(encoding="utf-8")
lock_text = (root / "uv.lock").read_text(encoding="utf-8")

# The dependency line in pyproject.toml looks like scikit-learn==1.7.2.
project_match = re.search(r"scikit-learn==(\d+\.\d+\.\d+)", project_text)
# In uv.lock, the [[package]] block puts version on the line after name.
lock_match = re.search(
    r'name = "scikit-learn"\s+version = "(\d+\.\d+\.\d+)"',
    lock_text,
)
options = ["1.6.1", "1.7.2", "1.8.0", "2.0.0"]
pinned = project_match.group(1) if project_match else None
lock_version = lock_match.group(1) if lock_match else None

# print: whether both files name the same version, and whether that version is one of the choices.
print("pyproject", pinned)
print("uv.lock", lock_version)
print("choice", pinned if pinned in options and pinned == lock_version else None)


pyproject 1.7.2
uv.lock 1.7.2
choice 1.7.2


## Q3 — load the model

**Question:** Load `pipeline.bin` with `pickle` and call `predict_proba` for this paid-ads lead. Report the probability of conversion rounded to three decimal places. Absolute error at most `0.005`. There is no option list.

```json
{
  "lead_source": "paid_ads",
  "industry": "technology",
  "employment_status": "employed",
  "location": "north_america",
  "number_of_courses_viewed": 2,
  "annual_income": 79276.0,
  "interaction_count": 4,
  "lead_score": 0.41
}
```

**Output:**

```text
0.532994
```

Rounded to 3 digits: **0.533**.

**Answer:** **0.533**


In [4]:
# import pickle: turn pipeline.bin back into a Python object.
import pickle
# Path: location of the model file.
from pathlib import Path

model_path = Path("pipeline.bin")
# open rb: open the binary file for reading.
with model_path.open("rb") as model_file:
    pipeline = pickle.load(model_file)

# Question 3 lead. The keys are the feature names used in training.
lead_q3 = {
    "lead_source": "paid_ads",
    "industry": "technology",
    "employment_status": "employed",
    "location": "north_america",
    "number_of_courses_viewed": 2,
    "annual_income": 79276.0,
    "interaction_count": 4,
    "lead_score": 0.41,
}
# predict_proba: class probabilities. [0, 1] is the conversion (class 1) probability.
probability_q3 = float(pipeline.predict_proba([lead_q3])[0, 1])
# round 3: the form wants three decimal places. Absolute tolerance is 0.005.
print(round(probability_q3, 3))
print("raw", probability_q3)


E:\IT_SPACES\AI\ZoomCamp\ML\.venv\Lib\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to unpickle estimator DictVectorizer from version 1.7.2 when using version 1.9.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
E:\IT_SPACES\AI\ZoomCamp\ML\.venv\Lib\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to unpickle estimator LogisticRegression from version 1.7.2 when using version 1.9.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
E:\IT_SPACES\AI\ZoomCamp\ML\.venv\Lib\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to unpickle estimator Pipeline from version 1.7.2 when using version 1.9.0. Th

0.533
raw 0.5329944186432507


## Q4 — serve the model

**Question:** Score this organic-search lead and report `conversion_probability` rounded to three decimal places. The same `0.005` absolute tolerance applies. The response must contain a probability in `[0, 1]` and a boolean `conversion` field.

```json
{
  "lead_source": "organic_search",
  "industry": "technology",
  "employment_status": "employed",
  "location": "europe",
  "number_of_courses_viewed": 4,
  "annual_income": 80304.0,
  "interaction_count": 7,
  "lead_score": 0.74
}
```

**Output:**

```text
0.769799
```

Rounded to 3 digits: **0.770**.

**Answer:** **0.770**


In [5]:
# import requests: send HTTP requests to the local API.
import requests

# url: POST /predict on the uvicorn server (port 9696).
url = "http://localhost:9696/predict"

# Question 4 lead. Same fields the API expects in the JSON body.
lead_q4 = {
    "lead_source": "organic_search",
    "industry": "technology",
    "employment_status": "employed",
    "location": "europe",
    "number_of_courses_viewed": 4,
    "annual_income": 80304.0,
    "interaction_count": 7,
    "lead_score": 0.74,
}
# requests.post: send the lead as JSON. response is the HTTP reply.
response = requests.post(url, json=lead_q4)
# response.json: parse the JSON body into a dict.
result = response.json()
# conversion_probability: score in [0, 1]. conversion: boolean at threshold 0.5.
prob = result.get("conversion_probability")
conv = result.get("conversion")
# round 3: the form wants three decimal places. Absolute tolerance is 0.005.
print("rounded probability:", round(prob, 3))
print("raw probability:", prob)
print("conversion:", conv)

rounded probability: 0.77
raw probability: 0.769799
conversion: True


## Q5 — container base image

**Question:** Which Python base-image tag is declared in the Dockerfile?

- `python:3.9-slim-bullseye`
- `python:3.11.15-slim-bookworm`
- `python:3.12-slim-bookworm`
- `python:3.13.10-slim-bookworm`

This question checks a version-controlled declaration. The displayed size of a local Docker image is not graded.

**Answer:** **python:3.11.15-slim-bookworm**


In [7]:
from pathlib import Path

# read_text: the first line is FROM python:...@sha256:...
dockerfile = (Path("Dockerfile")).read_text(encoding="utf-8")
from_line = dockerfile.splitlines()[0]
# split: keep the tag and drop the digest after @sha256.
image = from_line.split()[1].split("@")[0]
options = [
    "python:3.9-slim-bullseye",
    "python:3.11.15-slim-bookworm",
    "python:3.12-slim-bookworm",
    "python:3.13.10-slim-bookworm",
]
# print: the answer is the tag that matches a choice. Local image size is not graded.
print(image)
print("choice", image if image in options else None)


python:3.11.15-slim-bookworm
choice python:3.11.15-slim-bookworm


## Q6 — run the container

**Question:** Run the same request from Question 4 against the container. Report `conversion_probability` rounded to three decimal places. The expected result is the same reference inference as Question 4, with the same `0.005` absolute tolerance. Also verify that `/health` returns `{"status":"ok", ...}` and the model checksum recorded in `model_metadata.json`.

Build and run the image in a terminal from `HW_05` first (this blocks that terminal):

```bash
docker build -t zoomcamp-model:2026-hw5 .
docker run --rm -p 9696:9696 zoomcamp-model:2026-hw5
```

Then either run `python q6_test.py` in another terminal, or run the cell below against `localhost:9696`.

**Output:**

```text
0.770
```

**Answer:** **0.770**


In [8]:
import requests

# localhost:9696 must already be the running container (docker run).
url_predict = "http://localhost:9696/predict"
url_health = "http://localhost:9696/health"

# same lead as Question 4 / q6_test.py
lead_q6 = {
    "lead_source": "organic_search",
    "industry": "technology",
    "employment_status": "employed",
    "location": "europe",
    "number_of_courses_viewed": 4,
    "annual_income": 80304.0,
    "interaction_count": 7,
    "lead_score": 0.74,
}

# post: live container /predict, not a local pickle load
response_predict = requests.post(url_predict, json=lead_q6)
result = response_predict.json()
prob = result.get("conversion_probability")
# print: three decimals for the form; raw float to compare with Q4
print("rounded probability:", round(prob, 3))
print("raw probability:", prob)

# get: /health should include status ok and model_sha256
response_health = requests.get(url_health)
print("health response:", response_health.json())

rounded probability: 0.77
raw probability: 0.769799
health response: {'status': 'ok', 'model_sha256': '1646bbdcd38d4f044da6b630c5b332c93a314245a8c21929011c42de51f629f1', 'feature_defaults_sha256': '1694f842b908fac9f53c0cc19be6d91f934befc63af23cc32fefafde69acd0cd'}
